# Hidden Markov Model for Subject movement timeseries

Find hidden states in your data! Below is optimized for movement data from Anymaze files, but this is easily manipulated to work with whatever timeseries data you have.


## 0) Concepts

The HMM is based on augmenting the Markove chain, which is a model that tells us something about the probabilities of seuences of random variables (states), each of which can take on values from some set.

A Markov chain makes a very strong **assumption** that if we want to predict the future in the sequence, all that matters is the current state. It's as if to predict tomorrow's weaather but you weren't allowed to look at yesterday's weather.

Not all events are directly observable (**hidden**), and intead must be infered from what we can observe (**features**). Hidden Markov Models (HMMs) give us a way to infer the hidden structure behind such processes, modeling systems where what we see (the observations) is only a reflection of an underlying, unseen sequence of states.

Read more: [Daniel Jurafsky & James H. Martin, 2026](https://web.stanford.edu/~jurafsky/slp3/A.pdf) </br>
Code walkthrough: [BYU Applied and Computational Mathematics Program](https://labs.acme.byu.edu/Volume3/HMM/HMM.html)

### The Analogy (continued) ☔
Imagine you're locked in a room with no windows. Every day, a friend walks in carrying either an umbrella or not. You can't see outside. You want to figure out what the weather is like.
You notice patterns:
* When your friend brings an umbrella several days in a row, it's probably been raining
* Sunny days tend to cluster together too
* It's rare to go directly from a week of rain to a week of sun with no transition

The weather is the thing you care about but can't directly observe — it's hidden. </br>
The umbrella is what you can observe — it's the visible signal.

An HMM is a mathematical framework for this exact situation:
* There are hidden states (weather: sunny, rainy)
* You observe emissions (umbrella: yes, no) that are caused by those states
* States follow transition rules (sunny days tend to follow sunny days)
* You use the emissions to infer which hidden state you were probably in

### Key Terms

| Term              | Plain English                                                               |
| ----------------- | --------------------------------------------------------------------------- |
| KMeans            | Sort data points into K groups by geometric closeness                       |
| Centroid          | The average (center point) of a cluster                                     |
| Warm start        | Give an algorithm a sensible starting point instead of a random one         |
| Hidden state      | The thing you want to know but can't directly measure                       |
| Emission          | What you actually observe, caused by the hidden state                       |
| Transition matrix | A table of probabilities: given state A now, how likely is each state next? |
| Gaussian emission | Model each state's observations as a bell curve                             |
| Baum-Welch        | The learning algorithm that finds the best HMM parameters                   |
| Viterbi           | The decoding algorithm that finds the most likely state sequence            |
| Bout              | A continuous run of the same state                                          |
| Local optimum     | A solution that looks good nearby but isn't globally best                   |


In [ ]:
# Imports

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hmmlearn import hmm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.stats import sem

plt.rcParams["figure.dpi"] = 120


## 2) Configuration

Edit only this cell first. The rest of the notebook reads from these values.

`BIN_S` is the duration of one row in seconds. Keep it aligned with how the parquet (data file) was created.


In [ ]:
# Path to the parquet produced by HMM_analysis.py
PARQUET_PATH = r"X"

# All outputs (CSVs and figures) are written here
OUTPUT_DIR = r"X"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Hidden state count for the main fit
N_STATES = 4

# Duration of one row / bin in seconds
BIN_S = 0.1

# These labels apply after states are re-ranked by mean displacement per 100ms bin
STATE_LABELS = {
    0: "Slow movement",
    1: "near-wall immobility",
    2: "peripheral immobility",
    3: "Vigorous locomotion",
}

STATE_COLORS = {
    0: "#c6dbef",
    1: "#9ecae1",
    2: "#6baed6",
    3: "#2171b5",
}

# Sequence boundaries for the HMM
GROUP_COLS = ["subject", "cohort_id", "day", "context", "session"]
SORT_COLS = GROUP_COLS + ["time_s"]

# Features used by the HMM emission model
features = ["displacement_per_100ms", "abs_delta_displacement_per_100ms", "turning_angle", "dist_from_centre"]

# Numerical floor for diagonal variances after scaling
MIN_COVAR = 1e-3


## 3) Load the parquet and sort rows into true sequence order

This is the main structural fix. The HMM needs each session to be contiguous in time. If the parquet was already sorted, this step does nothing harmful; if it was not, this prevents broken transition estimates.


In [ ]:
df = pd.read_parquet(PARQUET_PATH)
print("Raw shape:", df.shape)
print("Columns:", df.columns.tolist())

# Sort so each independent sequence is contiguous and time is increasing inside it
for col in SORT_COLS:
    if col not in df.columns:
        raise KeyError(f"Missing required column: {col}")

df = df.sort_values(SORT_COLS).reset_index(drop=True)

# Basic sequence-length check
lengths = df.groupby(GROUP_COLS, sort=False).size().to_numpy()
assert lengths.sum() == len(df), "lengths mismatch after sorting"
print(f"{len(lengths)} sequences | min {lengths.min()} rows | max {lengths.max()} rows")
print(df.groupby(GROUP_COLS, sort=False).size().head().to_string())


# santiy check: grouping
# should print FALSE. If it prints TRUE, the grouping is incorrect.
# check that time resets properly at sequence boundaries
# explicit sanity check to confirm that rows are in the correct temporal order
# within each group before fitting
df["time_diff"] = df.groupby(GROUP_COLS)["time_s"].diff()
print("\nSanity Check: correct grouping")
print("Any negative time jumps within sequences:",
      (df["time_diff"] < 0).any())

df.drop(columns="time_diff", inplace=True)

## 4) Build the feature matrix and scale it




In [ ]:
missing = df[features].isna().sum()
if missing.any():
    print("Missing values per feature:\n", missing)
    df = df.dropna(subset=features).reset_index(drop=True)
    lengths = df.groupby(GROUP_COLS, sort=False).size().to_numpy()
    print("Dropped rows with missing features. New shape:", df.shape)

X = df[features].to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pd.DataFrame({"feature": features, "mean": scaler.mean_, "scale": scaler.scale_}).to_csv(
    os.path.join(OUTPUT_DIR, "scaler_params.csv"), index=False
)

print(pd.DataFrame({"feature": features, "mean": scaler.mean_, "scale": scaler.scale_}).to_string(index=False))


## 5) Optional sanity check: feature overlap

This is a quick check, not a decision rule. It helps confirm whether the main problem is redundancy or something else.


In [ ]:
corr = pd.DataFrame(df[features]).corr()
print(corr.to_string())


## 6) Helper for a stable HMM initialization

The original notebook initialized the transition matrix as uniform and estimated the starting probabilities from all KMeans labels pooled together. That is weak for sequence data. This helper uses the first label in each sequence for `startprob_`, and transitions only within each sequence for `transmat_`.


In [ ]:
def initialize_hmm(X_scaled, lengths, n_states, min_covar=1e-3, random_state=42):
    kmeans = KMeans(n_clusters=n_states, random_state=random_state, n_init=10)
    kmeans_labels = kmeans.fit_predict(X_scaled)

    # Start probabilities: first label from each independent sequence
    starts = np.cumsum(np.r_[0, lengths[:-1]])
    start_labels = kmeans_labels[starts]
    startprob = np.bincount(start_labels, minlength=n_states).astype(float)
    startprob = np.maximum(startprob, 1e-6)
    startprob /= startprob.sum()

    # Transition matrix: empirical transitions within each sequence, with light smoothing
    trans_counts = np.full((n_states, n_states), 1e-2)
    offset = 0
    for L in lengths:
        seq = kmeans_labels[offset:offset + L]
        if len(seq) > 1:
            for a, b in zip(seq[:-1], seq[1:]):
                trans_counts[a, b] += 1.0
        offset += L
    transmat = trans_counts / trans_counts.sum(axis=1, keepdims=True)

    # Diagonal covariances from KMeans groups, with a floor to avoid collapse
    covars = np.zeros((n_states, X_scaled.shape[1]))
    global_var = np.var(X_scaled, axis=0)
    for s in range(n_states):
        xs = X_scaled[kmeans_labels == s]
        covars[s] = np.var(xs, axis=0) if len(xs) > 1 else global_var
    covars = np.maximum(covars, min_covar)

    model = hmm.GaussianHMM(
        n_components=n_states,
        covariance_type="diag",
        n_iter=300,
        tol=1e-3,
        min_covar=min_covar,
        init_params="",
        params="stmc",
        random_state=random_state,
    )
    model.startprob_ = startprob
    model.transmat_ = transmat
    model.means_ = kmeans.cluster_centers_
    model.covars_ = covars
    return model, kmeans_labels


## 7) Fit the main HMM

This is the cell that should now behave much more reliably. The sequence order is fixed, and the initialization is sequence-aware.


In [ ]:
model, kmeans_labels = initialize_hmm(
    X_scaled=X_scaled,
    lengths=lengths,
    n_states=N_STATES,
    min_covar=MIN_COVAR,
    random_state=42,
)

model.fit(X_scaled, lengths=lengths)
df["hmm_state"] = model.predict(X_scaled, lengths=lengths)

print("Converged:", model.monitor_.converged)
print("Log-likelihood:", round(model.score(X_scaled, lengths=lengths), 2))
print("Transition matrix:\n", np.round(model.transmat_, 3))


## 8) Inspect the raw states, then rank them by displacement per 100ms bin

The raw HMM state labels are arbitrary. Ranking by mean displacement per 100ms bin makes the output easier to interpret and keeps the plots consistent across runs.


In [ ]:
state_summary = (
    df.groupby("hmm_state")[features]
      .mean()
      .reset_index()
)
state_summary["n_samples"] = df["hmm_state"].value_counts().sort_index().values
state_summary["pct_freezing"] = (
    df.groupby("hmm_state")["freezing"].mean().mul(100.0).to_numpy(dtype=float, na_value=np.nan)
    if "freezing" in df.columns
    else np.nan
)
state_summary["pct_in_platform"] = (
    df.groupby("hmm_state")["in_platform"].mean().mul(100.0).to_numpy(dtype=float, na_value=np.nan)
    if "in_platform" in df.columns
    else np.nan
)

state_summary = state_summary.sort_values("displacement_per_100ms").reset_index(drop=True)
rank_map = {int(state_summary.loc[i, "hmm_state"]): i for i in range(len(state_summary))}

df["hmm_state_ranked"] = df["hmm_state"].map(rank_map)
df["hmm_label"] = df["hmm_state_ranked"].map(STATE_LABELS)

state_summary["ranked_state"] = state_summary["hmm_state"].map(rank_map)
state_summary = state_summary.sort_values("ranked_state")
state_summary.to_csv(os.path.join(OUTPUT_DIR, "state_summary.csv"), index=False)
df.to_parquet(os.path.join(OUTPUT_DIR, "data_with_hmm_states.parquet"), index=False)

print(state_summary.to_string(index=False))
print("Rank map (raw -> ranked):", rank_map)


## 9) Plot feature means by ranked state and the transition matrix

These are the two plots that usually tell you whether the model has learned something meaningful or just shuffled labels.


In [ ]:
# Feature means with SEM by ranked state
summary_rows = []
for state in sorted(df["hmm_state_ranked"].dropna().unique()):
    mask = df["hmm_state_ranked"] == state
    for feat in features:
        vals = df.loc[mask, feat].dropna()
        summary_rows.append({
            "state": int(state),
            "feature": feat,
            "mean": vals.mean(),
            "sem": sem(vals) if len(vals) > 1 else 0.0,
        })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(OUTPUT_DIR, "feature_mean_sem_by_state.csv"), index=False)

for feat in features:
    fig, ax = plt.subplots(figsize=(6, 4))
    sub = summary_df[summary_df["feature"] == feat].sort_values("state")
    x = np.arange(len(sub))
    ax.bar(
        x,
        sub["mean"],
        yerr=sub["sem"],
        capsize=4,
        color=[STATE_COLORS[int(s)] for s in sub["state"]],
        edgecolor="black",
        linewidth=0.5,
    )
    ax.set_xticks(x)
    ax.set_xticklabels([STATE_LABELS[int(s)] for s in sub["state"]], rotation=25, ha="right")
    ax.set_ylabel(feat)
    ax.set_title(f"{feat}: mean ± SEM by state")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.show()

# Transition matrix in ranked-state order
order = [k for k, _ in sorted(rank_map.items(), key=lambda kv: kv[1])]
T = model.transmat_[np.ix_(order, order)]
labels = [STATE_LABELS[i] for i in range(len(order))]

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(T, vmin=0, vmax=1, cmap="Blues")
plt.colorbar(im, ax=ax, label="Transition probability")
ax.set_xticks(range(N_STATES))
ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
ax.set_yticks(range(N_STATES))
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel("To state")
ax.set_ylabel("From state")
ax.set_title("HMM Transition Matrix")
for i in range(N_STATES):
    for j in range(N_STATES):
        ax.text(j, i, f"{T[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.tight_layout()
plt.show()


## 10) Bout analysis

A bout is a continuous run of the same ranked state within a session. This section measures how long each state lasts and how often it appears.


In [ ]:
df = df.sort_values(SORT_COLS).reset_index(drop=True)

df["new_bout"] = (
    (df["hmm_state_ranked"] != df["hmm_state_ranked"].shift()) |
    (df["session"] != df["session"].shift()) |
    (df["subject"] != df["subject"].shift())
)
df["bout_id"] = df["new_bout"].cumsum()

bout_table = (
    df.groupby(["subject", "treatment", "context", "day", "session", "hmm_state_ranked", "bout_id"])
      .size()
      .reset_index(name="n_bins")
)
bout_table["bout_duration_s"] = bout_table["n_bins"] * BIN_S
bout_table.to_csv(os.path.join(OUTPUT_DIR, "state_bouts_raw.csv"), index=False)

print(bout_table.head(10).to_string(index=False))

# Collapse to one row per subject/day/session/state
# This makes daily summaries and plots easier to interpret.
daily_summary = (
    bout_table
    .groupby(["subject", "treatment", "context", "day", "session", "hmm_state_ranked"])
    .agg(
        total_duration_s=("bout_duration_s", "sum"),
        n_bouts=("bout_id", "count"),
        mean_bout_s=("bout_duration_s", "mean"),
    )
    .reset_index()
)
daily_summary.to_csv(os.path.join(OUTPUT_DIR, "bout_analysis.csv"), index=False)

daily_summary["day_num"] = daily_summary["day"].astype(str).str.extract(r"(\d+)").astype(int)
print(daily_summary.head(10).to_string(index=False))


## 11) Daily bout plots

These plots show whether each state becomes more frequent or longer across days.


In [ ]:
agg_bouts = (
    daily_summary
    .groupby(["day_num", "hmm_state_ranked"])["n_bouts"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
agg_bouts["sem"] = agg_bouts["std"] / np.sqrt(agg_bouts["count"])

fig, ax = plt.subplots(figsize=(8, 5))
for state in sorted(agg_bouts["hmm_state_ranked"].unique()):
    sub = agg_bouts[agg_bouts["hmm_state_ranked"] == state].sort_values("day_num")
    ax.plot(sub["day_num"], sub["mean"], marker="o",
            label=STATE_LABELS.get(int(state), f"State {state}"),
            color=STATE_COLORS.get(int(state), None))
    ax.fill_between(sub["day_num"], sub["mean"] - sub["sem"], sub["mean"] + sub["sem"],
                    alpha=0.2, color=STATE_COLORS.get(int(state), None), linewidth=0)
ax.set_xlabel("Day")
ax.set_ylabel("Number of bouts (mean ± SEM)")
ax.set_title("Bout frequency per state per day")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

agg_dur = (
    daily_summary
    .groupby(["day_num", "hmm_state_ranked"])["mean_bout_s"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
agg_dur["sem"] = agg_dur["std"] / np.sqrt(agg_dur["count"])

fig, ax = plt.subplots(figsize=(8, 5))
for state in sorted(agg_dur["hmm_state_ranked"].unique()):
    sub = agg_dur[agg_dur["hmm_state_ranked"] == state].sort_values("day_num")
    ax.plot(sub["day_num"], sub["mean"], marker="o",
            label=STATE_LABELS.get(int(state), f"State {state}"),
            color=STATE_COLORS.get(int(state), None))
    ax.fill_between(sub["day_num"], sub["mean"] - sub["sem"], sub["mean"] + sub["sem"],
                    alpha=0.2, color=STATE_COLORS.get(int(state), None), linewidth=0)
ax.set_xlabel("Day")
ax.set_ylabel("Mean bout duration, s (mean ± SEM)")
ax.set_title("Mean bout duration per state per day")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

# Proportion of total time spent in each state by day
daily_props = (
    daily_summary
    .groupby(["subject", "day_num", "hmm_state_ranked"])["total_duration_s"]
    .sum()
    .reset_index()
)
daily_props["pct"] = (
    daily_props["total_duration_s"]
    / daily_props.groupby(["subject", "day_num"])["total_duration_s"].transform("sum")
)
mean_props = (
    daily_props
    .groupby(["day_num", "hmm_state_ranked"])["pct"]
    .mean()
    .unstack(fill_value=0)
    .sort_index()
)
mean_props.to_csv(os.path.join(OUTPUT_DIR, "daily_state_proportions.csv"))

fig, ax = plt.subplots(figsize=(8, 5))
bottom = np.zeros(len(mean_props))
for state in sorted(mean_props.columns):
    ax.bar(mean_props.index, mean_props[state], bottom=bottom,
           label=STATE_LABELS.get(int(state), f"State {state}"),
           color=STATE_COLORS.get(int(state), None), edgecolor="black", linewidth=0.5)
    bottom += mean_props[state].to_numpy()
ax.set_xlabel("Day")
ax.set_ylabel("Proportion of time")
ax.set_title("Daily state composition")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()


## 12) Context occupancy

This is the main comparison for fear-conditioning style experiments: how much time each subject spends in each state in each context, split by treatment.


In [ ]:
if "context" in df.columns and "treatment" in df.columns:
    counts = (
        df.groupby(["subject", "treatment", "context", "hmm_state_ranked"])
          .size().reset_index(name="n_bins")
    )
    totals = (
        df.groupby(["subject", "treatment", "context"])
          .size().reset_index(name="total")
    )
    occ = counts.merge(totals, on=["subject", "treatment", "context"])
    occ["pct_time"] = occ["n_bins"].astype(float).div(occ["total"].astype(float)).mul(100.0)
    occ.to_csv(os.path.join(OUTPUT_DIR, "context_state_occupancy.csv"), index=False)

    contexts = sorted(occ["context"].dropna().unique())
    treatments = sorted(occ["treatment"].dropna().unique())
    states = sorted(occ["hmm_state_ranked"].dropna().unique())

    fig, axes = plt.subplots(1, len(states), figsize=(4 * len(states), 5), sharey=False)
    if len(states) == 1:
        axes = [axes]

    for ax, state in zip(axes, states):
        sub = occ[occ["hmm_state_ranked"] == state]
        width = 0.8 / max(len(treatments), 1)
        for i, trt in enumerate(treatments):
            t = sub[sub["treatment"] == trt]
            means = [t.loc[t["context"] == c, "pct_time"].mean() for c in contexts]
            x = np.arange(len(contexts)) + i * width - 0.4 + width / 2
            ax.bar(x, means, width=width, label=str(trt), edgecolor="black", linewidth=0.5)
        ax.set_xticks(np.arange(len(contexts)))
        ax.set_xticklabels(contexts, rotation=20, ha="right")
        ax.set_title(STATE_LABELS.get(int(state), f"State {state}"))
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        if ax is axes[0]:
            ax.set_ylabel("% time")
        ax.legend(frameon=False)

    plt.tight_layout()
    plt.show()
else:
    print("Skipping context occupancy because 'context' or 'treatment' is missing.")


## 13) BIC sweep for choosing the number of states

Run this only after the main pipeline is stable. Lower BIC is better. Look for the elbow, not the absolute minimum if the curve is flat.

However, BIC will often keep decreasing with large datasets. So, the practical decision is not "what minimizes BIC?" but "What is the smallest model that is interpretable and stable?".


In [ ]:
def fit_bic_for_n_states(n, X_scaled, lengths, min_covar=1e-3, random_state=42):
    model_n, _ = initialize_hmm(
        X_scaled=X_scaled,
        lengths=lengths,
        n_states=n,
        min_covar=min_covar,
        random_state=random_state,
    )
    model_n.fit(X_scaled, lengths=lengths)
    ll = model_n.score(X_scaled, lengths=lengths)
    n_features = X_scaled.shape[1]
    n_params = (n - 1) + n * (n - 1) + n * n_features + n * n_features
    bic = -2 * ll + n_params * np.log(len(X_scaled))
    return ll, bic

for n in range(2, 8):
    ll, bic = fit_bic_for_n_states(n, X_scaled, lengths, min_covar=MIN_COVAR, random_state=42)
    print(f"n_states={n}  log_likelihood={ll:.1f}  BIC={bic:.1f}")


In [ ]:
states = model.predict(X_scaled, lengths=lengths)
print(pd.Series(states).value_counts(normalize=True))

print(pd.DataFrame(model.means_, columns=features))

In [ ]:
# Percentage of each state that overlaps with freezing / in_platform
validation = (
    df.groupby("hmm_state_ranked")[["freezing", "in_platform"]]
    .mean()
    .mul(100.0)
)
validation.columns = ["pct_freezing", "pct_in_platform"]
validation.index = [STATE_LABELS[i] for i in validation.index]
print(validation.round(1))

In [ ]:
# Of all freezing frames, how are they distributed across HMM states?
freezing_rows = df[df["freezing"] == True]
print(
    freezing_rows["hmm_state_ranked"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
    .rename(STATE_LABELS)
)

In [ ]:
print(df.groupby(["treatment", "hmm_state_ranked"]).size().unstack(fill_value=0))
print(df.groupby(["treatment", "day", "context"]).size())